In [12]:
import pandas as pd 
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from  lightgbm import LGBMRegressor
from sklearn.metrics import mean_squared_error , mean_absolute_error , r2_score
import numpy as np
from sklearn.tree import DecisionTreeRegressor
import time 

In [13]:
venta_carros = pd.read_csv(r"C:\Users\usuario1\Documents\car_data.csv")
venta_carros.info()
print(venta_carros.isnull().sum())
print(venta_carros.sample(10))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 354369 entries, 0 to 354368
Data columns (total 16 columns):
 #   Column             Non-Null Count   Dtype 
---  ------             --------------   ----- 
 0   DateCrawled        354369 non-null  object
 1   Price              354369 non-null  int64 
 2   VehicleType        316879 non-null  object
 3   RegistrationYear   354369 non-null  int64 
 4   Gearbox            334536 non-null  object
 5   Power              354369 non-null  int64 
 6   Model              334664 non-null  object
 7   Mileage            354369 non-null  int64 
 8   RegistrationMonth  354369 non-null  int64 
 9   FuelType           321474 non-null  object
 10  Brand              354369 non-null  object
 11  NotRepaired        283215 non-null  object
 12  DateCreated        354369 non-null  object
 13  NumberOfPictures   354369 non-null  int64 
 14  PostalCode         354369 non-null  int64 
 15  LastSeen           354369 non-null  object
dtypes: int64(7), object(

In [14]:
print('Rango de precios original :', venta_carros['Price'].min() ,'-' , venta_carros['Price'].max())
print('Rango de años de carros ', venta_carros['RegistrationYear'].min() , '-' , venta_carros['RegistrationYear'].max())
print('Rango de potencia de carros', venta_carros['Power'].min(), '-', venta_carros['Power'].max())
años_validos = venta_carros[(venta_carros['RegistrationYear'] >= 1980) & (venta_carros['RegistrationYear'] <= 2016)]
print(f"Autos con años validos {len(años_validos)}")
autos_precio_0 = venta_carros[venta_carros ['Price'] == 0 ]
print(f'Autos con precio 0 {len(autos_precio_0)}')
print('Caracteristricas de autos con precio 0')
print(f"Años mas comunes : {autos_precio_0['RegistrationYear'].value_counts().head()}")
print(f"Autos con precio de 20000 : {(venta_carros['Price'] == 20000).sum()}")

 

Rango de precios original : 0 - 20000
Rango de años de carros  1000 - 9999
Rango de potencia de carros 0 - 20000
Autos con años validos 336387
Autos con precio 0 10772
Caracteristricas de autos con precio 0
Años mas comunes : RegistrationYear
2000    1418
1999     733
1998     721
1997     675
1995     654
Name: count, dtype: int64
Autos con precio de 20000 : 268


In [15]:
print('Limpieza de datos')
print('Registros originales :' , len(venta_carros))
venta_carros_limpio = años_validos[(años_validos['Price'] >= 500) & (años_validos['Price'] <= 20000)]
venta_carros_limpio = venta_carros_limpio[(venta_carros_limpio['Power'] >= 30) & (venta_carros_limpio['Power'] <= 500)]
columnas_importantes = ['VehicleType', 'Gearbox','Model', 'FuelType' ]
venta_carros_limpio = venta_carros_limpio.dropna(subset = columnas_importantes)
print('Registros despues de la limpieza :', len(venta_carros_limpio))
print(f'Porcentage de registros conservados:  {(len(venta_carros_limpio)/ len(venta_carros))*100:.1f}%')


Limpieza de datos
Registros originales : 354369
Registros despues de la limpieza : 250015
Porcentage de registros conservados:  70.6%


## Entrenamiento del modelo 

In [ ]:
%%time
features = venta_carros_limpio.drop(['Price'], axis=1)
target= venta_carros_limpio['Price']
numeric_features =['RegistrationYear', 'Power', 'Mileage', 'RegistrationMonth']
categorical_features = ['VehicleType', 'Model', 'Brand', 'FuelType', 'Gearbox']
caracteristicas_escogidas = venta_carros_limpio [numeric_features + categorical_features]
features_train, features_test, target_train, target_test = train_test_split(caracteristicas_escogidas, target, test_size = 0.2, random_state = 45)
preprocessor = ColumnTransformer( transformers = [('num', 'passthrough', numeric_features),  ('cat', OneHotEncoder( sparse_output=    False, handle_unknown = 'ignore'), categorical_features)])
features_train_encoded = preprocessor.fit_transform(features_train)
features_test_encoded = preprocessor.transform(features_test)

modelo_linear = LinearRegression()
modelo_linear.fit(features_train_encoded, target_train)
lin_pred = modelo_linear.predict(features_test_encoded)

rmse = np.sqrt(mean_squared_error(target_test, lin_pred))
mae = mean_absolute_error(target_test, lin_pred)
r2 = r2_score(target_test, lin_pred)

print('Regresion lineal')
print(f'RMSE : {rmse:.2f} euros ')
print(f'MAE : {mae:.2f} euros')
print(f'R2 : {r2: .3f}')



Regresion lineal
RMSE : 2340.60 euros 
MAE : 1666.03 euros
R2 :  0.743
CPU times: total: 10.9 s
Wall time: 6.45 s


In [17]:
%%time
for n_est in [50, 100, 200] :
    model = RandomForestRegressor(random_state = 45 , n_estimators = n_est)
    model.fit(features_train_encoded, target_train)
    predic_for = model.predict(features_test_encoded)
    mae_for = mean_absolute_error(target_test, predic_for)
    rmse_for = np.sqrt(mean_squared_error(target_test, predic_for))
    r2_for = r2_score(target_test, predic_for)
    print(f'MAE RandomForestRegressor ({n_est} estimators) : {mae_for:.2f} euros')
    print(f'RMSE RandomForestRegressor ({n_est} estimators) : {rmse_for:.2f} euros')
    print(f'R2 RandomForestRegressor ({n_est} estimators) : {r2_for:.3f} ')
    

MAE RandomForestRegressor (50 estimators) : 972.22 euros
RMSE RandomForestRegressor (50 estimators) : 1554.70 euros
R2 RandomForestRegressor (50 estimators) : 0.886 
MAE RandomForestRegressor (100 estimators) : 969.58 euros
RMSE RandomForestRegressor (100 estimators) : 1550.09 euros
R2 RandomForestRegressor (100 estimators) : 0.887 
MAE RandomForestRegressor (200 estimators) : 967.97 euros
RMSE RandomForestRegressor (200 estimators) : 1547.14 euros
R2 RandomForestRegressor (200 estimators) : 0.888 
CPU times: total: 27min 38s
Wall time: 28min 24s


In [18]:
%%time
hiperparametros = [{'n_estimators': 100, 
                   'learning_rate': 0.1,
                   'max_depth' : 6, 
                   'num_leaves': 31, 
                   'verbose': -1},
                  {'n_estimators': 200,
                  'learning_rate': 0.05,
                  'max_depth': 10, 
                  'num_leaves': 50, 
                  'verbose': -1},
                  {'n_estimators': 300,
                  'learning_rate': 0.03,
                  'max_depth': 15,
                  'num_leaves': 100,
                  'verbose': -1}]
for params in hiperparametros:
    model_lgb = LGBMRegressor(random_state = 45 , **params)
    model_lgb.fit(features_train_encoded, target_train)
    lgb_predic = model_lgb.predict(features_test_encoded)
    mae_lgb = mean_absolute_error(target_test, lgb_predic)
    rmse_lgb = np.sqrt(mean_squared_error(target_test, lgb_predic))
    r2_lgb = r2_score(target_test, lgb_predic)
    print(f'MAE lightGBM (parametros {params} ): {mae_lgb:.2f} euros')
    print(f'RMSE LightGBM (parametros {params}) : {rmse_lgb:.2f} euros')
    print(f'R2 LightGBM (parametros {params}) : {r2_lgb:.3f}')

c:\Users\usuario1\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


MAE lightGBM (parametros {'n_estimators': 100, 'learning_rate': 0.1, 'max_depth': 6, 'num_leaves': 31, 'verbose': -1} ): 1107.38 euros
RMSE LightGBM (parametros {'n_estimators': 100, 'learning_rate': 0.1, 'max_depth': 6, 'num_leaves': 31, 'verbose': -1}) : 1679.71 euros
R2 LightGBM (parametros {'n_estimators': 100, 'learning_rate': 0.1, 'max_depth': 6, 'num_leaves': 31, 'verbose': -1}) : 0.867


c:\Users\usuario1\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


MAE lightGBM (parametros {'n_estimators': 200, 'learning_rate': 0.05, 'max_depth': 10, 'num_leaves': 50, 'verbose': -1} ): 1067.05 euros
RMSE LightGBM (parametros {'n_estimators': 200, 'learning_rate': 0.05, 'max_depth': 10, 'num_leaves': 50, 'verbose': -1}) : 1625.23 euros
R2 LightGBM (parametros {'n_estimators': 200, 'learning_rate': 0.05, 'max_depth': 10, 'num_leaves': 50, 'verbose': -1}) : 0.876


c:\Users\usuario1\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


MAE lightGBM (parametros {'n_estimators': 300, 'learning_rate': 0.03, 'max_depth': 15, 'num_leaves': 100, 'verbose': -1} ): 1028.77 euros
RMSE LightGBM (parametros {'n_estimators': 300, 'learning_rate': 0.03, 'max_depth': 15, 'num_leaves': 100, 'verbose': -1}) : 1571.10 euros
R2 LightGBM (parametros {'n_estimators': 300, 'learning_rate': 0.03, 'max_depth': 15, 'num_leaves': 100, 'verbose': -1}) : 0.884
CPU times: total: 15.2 s
Wall time: 12.8 s


In [19]:
def evaluar_modelo_completo(modelo, nombre, features_train, features_test, target_train, target_test):
    start_time = time.time()
    modelo.fit(features_train, target_train)
    training_time = time.time() - start_time

    predictions = modelo.predict(features_test)
    rmse = np.sqrt(mean_squared_error(target_test, predictions))
    mae = mean_absolute_error (target_test, predictions)
    r2 = r2_score(target_test, predictions)

    if training_time < 60 :
        tiempo_str = f'{training_time:.2f} s '
    else :
        tiempo_str = f'{training_time/60:.1f} min'

    return {'Modelo': nombre,
           'RMSE' : round(rmse, 2),
           'MAE': round(mae, 2),
           'R2': round(r2, 2),
           'Tiempo': tiempo_str}
    

In [20]:
%%time
for max_depth in [5, 10, 15] :
    model_tree = DecisionTreeRegressor(random_state = 45, max_depth = max_depth )
    model_tree.fit(features_train_encoded, target_train)
    tree_predict = model_tree.predict(features_test_encoded)
    mae_tree = mean_absolute_error(target_test, tree_predict)
    rmse_tree = np.sqrt(mean_squared_error(target_test, tree_predict))
    r2_tree = r2_score(target_test, tree_predict)
    print(f'MAE DesicionTreeRegressor ( depth {max_depth}) : {mae_tree:.2f} euros ')
    print(f'RMSE DesicionTreeRegressor ( depth {max_depth}) : {rmse_tree:.2f} euros')
    print(f'R2 DesicionTreeRegressor (depth {max_depth}) : {r2_tree:.3f}')

MAE DesicionTreeRegressor ( depth 5) : 1605.33 euros 
RMSE DesicionTreeRegressor ( depth 5) : 2302.62 euros
R2 DesicionTreeRegressor (depth 5) : 0.751
MAE DesicionTreeRegressor ( depth 10) : 1288.64 euros 
RMSE DesicionTreeRegressor ( depth 10) : 1906.98 euros
R2 DesicionTreeRegressor (depth 10) : 0.829
MAE DesicionTreeRegressor ( depth 15) : 1144.26 euros 
RMSE DesicionTreeRegressor ( depth 15) : 1784.91 euros
R2 DesicionTreeRegressor (depth 15) : 0.850
CPU times: total: 17 s
Wall time: 17.6 s


## Análisis del modelo

In [21]:
todos_resultados = []
modelos_a_evaluar = [(LinearRegression(), 'Regresion Lineal'),
                    (RandomForestRegressor(n_estimators = 200, random_state = 45), 'Random Forest (200)'),
                     (LGBMRegressor (n_estimators = 300, learning_rate = 0.03, max_depth = 15, num_leaves = 100, random_state = 45, verbose = -1), 'LightGBM (300)'),
                     (DecisionTreeRegressor(max_depth = 15, random_state = 45), 'Decision Tree (15)')
                    ]
print('Evaluando modelos')

for modelo, nombre in modelos_a_evaluar:
    print(f'Evaluando {nombre}')
    resultado = evaluar_modelo_completo(modelo, nombre, features_train_encoded, features_test_encoded, target_train, target_test)
    todos_resultados.append(resultado)
    print(f' {nombre} completo')

df_resultados = pd.DataFrame(todos_resultados)

print('Comparacion de modelos')
print(df_resultados.to_string(index = False))

Evaluando modelos
Evaluando Regresion Lineal
 Regresion Lineal completo
Evaluando Random Forest (200)
 Random Forest (200) completo
Evaluando LightGBM (300)


c:\Users\usuario1\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


 LightGBM (300) completo
Evaluando Decision Tree (15)
 Decision Tree (15) completo
Comparacion de modelos
             Modelo    RMSE     MAE   R2   Tiempo
   Regresion Lineal 2340.60 1666.03 0.74  5.10 s 
Random Forest (200) 1547.14  967.97 0.89 28.0 min
     LightGBM (300) 1571.10 1028.77 0.88  5.57 s 
 Decision Tree (15) 1784.91 1144.26 0.85 10.27 s 


In [ ]:
print('ANÁLISIS DE VELOCIDAD DE PREDICCIÓN ')

def medir_velocidad_prediccion(modelo, features_test, n_predicciones=1000):
    muestra = features_test[:n_predicciones]
    start_time = time.time()
    predicciones = modelo.predict(muestra)
    tiempo_total = time.time() - start_time
    tiempo_por_prediccion = (tiempo_total / n_predicciones) * 1000
    return tiempo_por_prediccion
    
print("Entrenando modelos para medir velocidad.")
modelos_entrenados = {}

for modelo, nombre in modelos_a_evaluar:
    modelo.fit(features_train_encoded, target_train)
    modelos_entrenados[nombre] = modelo

print("\nVELOCIDAD DE PREDICCIÓN:")
velocidades = []
for nombre, modelo in modelos_entrenados.items():
    velocidad = medir_velocidad_prediccion(modelo, features_test_encoded)
    velocidades.append({'Modelo': nombre, 'Velocidad (ms)': round(velocidad, 3)})
    print(f"{nombre}: {velocidad:.3f} ms por predicción")

df_velocidades = pd.DataFrame(velocidades)
print(" RESUMEN DE VELOCIDADES:")
print(df_velocidades.to_string(index=False))

ANÁLISIS DE VELOCIDAD DE PREDICCIÓN 
Entrenando modelos para medir velocidad.


Conclusiones

en conclusion en  términos de calidad de predicción, Random Forest obtuvo el mejor rendimiento, alcanzando el menor RMSE (1547), el menor MAE (967) y el mayor coeficiente de determinación R² (0.89), lo que indica una alta capacidad para explicar la variabilidad del precio de los vehículos. LightGBM mostró un rendimiento muy similar, con un RMSE de 1571 y un R² de 0.88, confirmando también una excelente capacidad predictiva. Por otro lado, la Regresión Lineal presentó el peor desempeño (RMSE de 2340 y R² de 0.74), lo que indica que los modelos lineales no son suficientes para capturar la complejidad de las relaciones presentes en los datos.

En cuanto a la velocidad de predicción, la Regresión Lineal fue el modelo más rápido debido a su simplicidad, seguida por el Árbol de Decisión y LightGBM. Random Forest presentó el mayor tiempo de predicción debido a su estructura basada en múltiples árboles. Respecto al tiempo de entrenamiento, Random Forest fue el modelo más lento, requiriendo aproximadamente 10.4 minutos, mientras que LightGBM logró un entrenamiento significativamente más rápido, manteniendo una calidad predictiva comparable.

el mejor modelo considerando el equilibrio entre precisión y velocidad de predicción es el  LightGBM ya que  representa la mejor opción para su implementación en la aplicación de Rusty Bargain. Este modelo ofrece una excelente calidad de predicción con un menor tiempo de entrenamiento y predicción en comparación con Random Forest, lo que lo hace más adecuado para un entorno de producción donde la rapidez y la eficiencia son factores críticos.

En conclusión, LightGBM proporciona el mejor balance entre rendimiento predictivo y eficiencia, cumpliendo de manera óptima los requisitos del proyecto.